# HITL-IoT v1.1 — Step 7
## Selective Deferral, Probability Calibration, and Human–AI Oversight Mechanisms

**Canonical notebook:** `04_HITL-IoT_v1.1_Selective_Deferral_Calibration_Oversight.ipynb`

This notebook is the next stage after the frozen leakage-safe nine-model Track-1 benchmark.

### Scientific objective

Step 7 tests whether **independently learned IDS probabilities** can support reproducible oversight mechanisms such as:

- probability calibration;
- selective prediction / reject option;
- uncertainty-based review allocation;
- Human–AI deferral under fixed review budgets;
- confidence-gated overrides;
- learned deferral utility;
- matched-budget random-routing baselines;
- correction-versus-harm analysis.

The primary analysis uses the **Random Forest reference classifier selected on temporal validation only in Step 6**.

### Scope discipline

The configured oversight profiles are **benchmark-generated variables**, not observations of real security analysts. Therefore, the results quantify **mechanism behaviour under explicit controlled assumptions**. They do not establish empirical human behaviour, analyst competence, SOC effectiveness, or regulatory compliance.

### Core safeguards

1. The frozen Step-6 test set is never used to fit calibration models, thresholds, routing policies, or learned defer models.
2. Calibration is fitted on an early half of temporal validation.
3. Policy thresholds are chosen on the later half of temporal validation.
4. Final results are reported on untouched temporal test.
5. `generated_attack_score` is never used as the AI confidence source.
6. The primary AI confidence source is the independently learned Random-Forest probability from Step 6.
7. Oversight-profile variables are used only in explicitly labelled Track-2 collaboration experiments.

8. The notebook refuses to use any `release_v1.1` folder whose dataset SHA-256 does not match the frozen canonical release.

## Methodological references

This notebook follows established ideas from:

- Guo et al. (2017), **On Calibration of Modern Neural Networks**, ICML/PMLR 70.
- Geifman and El-Yaniv (2019), **SelectiveNet: A Deep Neural Network with an Integrated Reject Option**, ICML/PMLR 97.
- Mozannar and Sontag (2020), **Consistent Estimators for Learning to Defer to an Expert**, ICML/PMLR 119.

The learned-deferral baseline implemented here is a transparent benchmark rejector inspired by learning-to-defer research; it is **not claimed to reproduce the exact Mozannar–Sontag loss**.

In [ ]:
from __future__ import annotations

import hashlib, json, math, os, platform, random, re, sys, time, warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, log_loss,
    matthews_corrcoef, precision_score, recall_score, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

EXPECTED_CONTEXT_FINGERPRINT = "69b553133c156aa9c37a45d69d0a75ec9dc2eafb29a128294df9ec35b5b299f5"
EXPECTED_DATASET_SHA256 = "c6dd623029f6ca0967773379b2afbba96b73ab9cb2028c5e81ef89a1b15e1291"
REFERENCE_MODEL = "Random Forest"

RUN_ALL9_CALIBRATION = True
RUN_IID_SENSITIVITY = True
RUN_LEARNED_DEFER = True

SELECTIVE_BUDGETS = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30]
OVERSIGHT_BUDGETS_FULL_TEST = [0.02, 0.04, 0.06, 0.08]
RANDOM_ROUTING_REPS = 30
ECE_BINS = 15

OUT_DIR = Path("step7_selective_deferral_calibration_oversight_v1.1")
for d in ["audit", "tables", "figures", "models", "probabilities", "manuscript_values", "logs"]:
    (OUT_DIR / d).mkdir(parents=True, exist_ok=True)

print("Step 7 configuration loaded.")

In [ ]:
# ---------------------------------------------------------------------------
# Publication figure system (identical conventions to the Step-6 notebook):
# journal sans-serif, 7.5-10 pt text, minimal spines, subtle grids, fixed
# colour-blind-safe semantics, vector PDF/SVG plus 600-dpi PNG, panel labels.
# ---------------------------------------------------------------------------
import re
import matplotlib as mpl
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, PercentFormatter, MaxNLocator

SINGLE_COL, DOUBLE_COL = 3.5, 7.16
SIGNAL_COLORS = {
    "RF raw uncertainty": "#0072B2",
    "RF calibrated uncertainty": "#009E73",
    "RF calibrated entropy": "#E69F00",
    "All-9 probability disagreement": "#CC79A7",
    "All-9 disagreement": "#CC79A7",
}
SIGNAL_MARKERS = {
    "RF raw uncertainty": "o", "RF calibrated uncertainty": "s", "RF calibrated entropy": "^",
    "All-9 probability disagreement": "D", "All-9 disagreement": "D",
}
POLICY_COLORS = {"unconditional profile override": "#D55E00",
                 "confidence-gated profile override": "#0072B2",
                 "oracle upper bound": "#4D4D4D",
                 "Matched-budget random routing": "#8C8C8C"}
OUTCOME_COLORS = {"raw": "#8C8C8C", "platt": "#0072B2", "isotonic": "#009E73"}
NEUTRAL, GRID = "#4D4D4D", "#E3E3E3"

def set_publication_style():
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
        "font.size": 8.5, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
        "legend.fontsize": 7.5, "legend.frameon": False, "legend.handlelength": 2.0,
        "axes.linewidth": 0.6, "axes.edgecolor": "#333333", "axes.spines.top": False,
        "axes.spines.right": False, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "xtick.major.size": 2.5, "ytick.major.size": 2.5, "lines.linewidth": 1.3,
        "figure.dpi": 110, "savefig.dpi": 600, "pdf.fonttype": 42, "ps.fonttype": 42,
        "svg.fonttype": "none", "axes.formatter.use_mathtext": False})

def ygrid(ax, axis="y"):
    ax.grid(True, axis=axis, color=GRID, linewidth=0.5); ax.set_axisbelow(True)

def add_panel_label(ax, letter, x=-0.16, y=1.04):
    ax.text(x, y, f"({letter})", transform=ax.transAxes, fontsize=10.5,
            fontweight="bold", va="bottom", ha="left")

FIGURE_REGISTRY = {}

def save_publication_figure(fig, stem, caption, n_panels):
    paths = {}
    for ext, kw in (("pdf", {}), ("svg", {}), ("png", {"dpi": 600})):
        p = OUT_DIR / "figures" / f"{stem}.{ext}"
        fig.savefig(p, bbox_inches="tight", pad_inches=0.03, **kw)
        paths[ext] = p
    fig.canvas.draw()
    FIGURE_REGISTRY[stem] = {
        "paths": paths, "caption": caption, "n_panels": n_panels,
        "size_in": tuple(np.round(fig.get_size_inches(), 2)),
        "axes": [{"xlabel": a.get_xlabel(), "ylabel": a.get_ylabel(),
                  "named_ticks": any(not re.fullmatch(r"[^A-Za-z]*", t.get_text() or "")
                                     for t in a.get_xticklabels() + a.get_yticklabels())}
                 for a in fig.axes if a.axison],
        "texts": " ".join(t.get_text() for a in fig.axes for t in [a.xaxis.label, a.yaxis.label, *a.texts])}
    return paths

set_publication_style()
thousands = FuncFormatter(lambda v, _: f"{v:,.0f}")
print("Publication figure style loaded:",
      mpl.font_manager.FontProperties(family=mpl.rcParams["font.sans-serif"]).get_name())


In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

def find_canonical_release(candidates, expected_dataset_sha256):
    checked = []
    for root in candidates:
        root = Path(root)
        dataset = root / "HITL-IoT_dataset_v1.1.csv"
        oversight = root / "HITL-IoT_oversight_subset_v1.1.csv"
        if dataset.exists() and oversight.exists():
            observed = sha256_file(dataset)
            checked.append((str(root), observed))
            if observed == expected_dataset_sha256:
                return root, checked
    return None, checked

# Optional manual overrides for unusual folder layouts.
STEP6_DIR_OVERRIDE = None
RELEASE_DIR_OVERRIDE = None
MODEL_DIR_OVERRIDE = None

step6_candidates = [
    Path("track1_benchmark_v1.1_final"),
    Path("track1_benchmark_v1.1_review_revision"),
    Path("parallel_review/track1_benchmark_v1.1_review_revision"),
]
STEP6_DIR = Path(STEP6_DIR_OVERRIDE) if STEP6_DIR_OVERRIDE else first_existing(step6_candidates)

release_candidates = [
    Path("data/v1.1"),
    Path("release_v1.1"),
    Path(".") / "release_v1.1",
    Path("release"),
    Path("data") / "release_v1.1",
]
if RELEASE_DIR_OVERRIDE:
    release_candidates.insert(0, Path(RELEASE_DIR_OVERRIDE))
RELEASE_DIR, RELEASE_CHECKS = find_canonical_release(release_candidates, EXPECTED_DATASET_SHA256)

model_candidates = [
    Path("track1_model_binaries_v1.1/models"),
    Path("parallel_bins/models"),
    (STEP6_DIR / "models") if STEP6_DIR else Path("__missing__"),
]
MODEL_DIR = Path(MODEL_DIR_OVERRIDE) if MODEL_DIR_OVERRIDE else first_existing(model_candidates)

assert STEP6_DIR is not None, "Step-6 result directory not found."
assert RELEASE_DIR is not None, (
    "Canonical frozen release_v1.1 not found. The notebook refuses to use a stale dataset. "
    f"Checked: {RELEASE_CHECKS}. Set RELEASE_DIR_OVERRIDE to the canonical folder if needed."
)
assert MODEL_DIR is not None, "Step-6 model-binary directory not found."

print("Step-6 results:", STEP6_DIR)
print("Canonical release:", RELEASE_DIR)
print("Model binaries:", MODEL_DIR)

In [ ]:
context_path = STEP6_DIR / "logs" / "benchmark_context.json"
reference_path = STEP6_DIR / "manuscript_values" / "reference_model_selection.json"
results_path = STEP6_DIR / "manuscript_values" / "track1_benchmark_results_all9.json"

context = json.loads(context_path.read_text(encoding="utf-8"))
reference = json.loads(reference_path.read_text(encoding="utf-8"))
step6_results = json.loads(results_path.read_text(encoding="utf-8"))

assert context["context_fingerprint"] == EXPECTED_CONTEXT_FINGERPRINT
assert reference["selected_model"] == REFERENCE_MODEL

dataset_path = RELEASE_DIR / "HITL-IoT_dataset_v1.1.csv"
oversight_path = RELEASE_DIR / "HITL-IoT_oversight_subset_v1.1.csv"
assert dataset_path.exists() and oversight_path.exists()
assert sha256_file(dataset_path) == EXPECTED_DATASET_SHA256

rf_model_path = MODEL_DIR / "temporal_random_forest.joblib"
rf_meta_path = MODEL_DIR / "temporal_random_forest_metadata.json"
assert rf_model_path.exists() and rf_meta_path.exists()
rf_meta = json.loads(rf_meta_path.read_text(encoding="utf-8"))
assert sha256_file(rf_model_path) == rf_meta["files"]["model"]["sha256"]

print("[PASS] Frozen Step-6 context verified.")
print("[PASS] Reference classifier:", reference["selected_model"])
print("[PASS] Frozen RF binary verified.")

In [ ]:
p_val_all = pd.read_csv(STEP6_DIR / "probabilities" / "temporal_validation_probabilities_all9.csv")
p_test_all = pd.read_csv(STEP6_DIR / "probabilities" / "temporal_test_probabilities_all9.csv")

expected_cols = {
    "flow_id", "timestamp", "evaluation_protocol", "split",
    "y_true", "model", "y_probability_attack", "y_pred_0p5"
}
assert expected_cols <= set(p_val_all.columns)
assert expected_cols <= set(p_test_all.columns)
assert p_val_all["model"].nunique() == 9 and p_test_all["model"].nunique() == 9
assert not p_val_all.duplicated(["flow_id", "model"]).any()
assert not p_test_all.duplicated(["flow_id", "model"]).any()
assert p_val_all["y_probability_attack"].between(0,1).all()
assert p_test_all["y_probability_attack"].between(0,1).all()

df = pd.read_csv(dataset_path)
oversight = pd.read_csv(oversight_path)
df["timestamp"] = pd.to_datetime(df["timestamp"])
oversight["timestamp"] = pd.to_datetime(oversight["timestamp"])

val_ids = set(p_val_all["flow_id"].unique())
test_ids = set(p_test_all["flow_id"].unique())
assert val_ids.isdisjoint(test_ids)

df["temporal_split_step7"] = np.where(
    df["flow_id"].isin(val_ids), "validation",
    np.where(df["flow_id"].isin(test_ids), "test", "train")
)
assert df.loc[df.temporal_split_step7=="train","timestamp"].max() < df.loc[df.temporal_split_step7=="validation","timestamp"].min()
assert df.loc[df.temporal_split_step7=="validation","timestamp"].max() < df.loc[df.temporal_split_step7=="test","timestamp"].min()

oversight = oversight.merge(df[["flow_id","temporal_split_step7"]], on="flow_id", how="left", validate="one_to_one")

print(df["temporal_split_step7"].value_counts())
print("\nOversight subset:")
print(oversight["temporal_split_step7"].value_counts())

## Validation discipline for Step 7

The temporal validation partition is divided chronologically into two non-overlapping halves:

- **Calibration-fit half:** fit Platt/sigmoid and isotonic calibration models.
- **Policy-validation half:** select calibration method, review thresholds, deferral thresholds, and confidence-gating settings.

The temporal test partition remains untouched until final evaluation.

In [ ]:
def model_prob(df_prob, model):
    x = df_prob[df_prob["model"] == model].copy()
    x["timestamp"] = pd.to_datetime(x["timestamp"])
    return x.sort_values(["timestamp","flow_id"]).reset_index(drop=True)

rf_val = model_prob(p_val_all, REFERENCE_MODEL)
rf_test = model_prob(p_test_all, REFERENCE_MODEL)

mid = len(rf_val)//2
cal_fit = rf_val.iloc[:mid].copy()
policy_val = rf_val.iloc[mid:].copy()

assert cal_fit["timestamp"].max() <= policy_val["timestamp"].min()
assert set(cal_fit.flow_id).isdisjoint(set(policy_val.flow_id))

print("Calibration-fit:", len(cal_fit), cal_fit.timestamp.min(), "→", cal_fit.timestamp.max())
print("Policy-validation:", len(policy_val), policy_val.timestamp.min(), "→", policy_val.timestamp.max())
print("Temporal test:", len(rf_test), rf_test.timestamp.min(), "→", rf_test.timestamp.max())

In [ ]:
EPS = 1e-7

def classification_metrics(y, p, threshold=0.5):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        # MCC and ranking metrics are undefined on degenerate subsets (single class present or predicted);
        # return NaN rather than a misleading 0.0.
        "mcc": matthews_corrcoef(y, pred) if (len(np.unique(pred)) > 1 and len(np.unique(y)) > 1) else np.nan,
        "auroc": roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
        "average_precision": average_precision_score(y, p) if len(np.unique(y)) > 1 else np.nan,
        "fpr": fp/(fp+tn) if (fp+tn) else np.nan,
        "fnr": fn/(fn+tp) if (fn+tp) else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }

def ece_score(y, p, n_bins=15):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    bins = np.linspace(0, 1, n_bins+1)
    idx = np.clip(np.digitize(p, bins, right=True)-1, 0, n_bins-1)
    ece = 0.0
    rows = []
    for b in range(n_bins):
        m = idx == b
        if not m.any():
            continue
        conf = p[m].mean()
        frac = y[m].mean()
        w = m.mean()
        ece += w * abs(conf-frac)
        rows.append({"bin":b,"n":int(m.sum()),"mean_probability":conf,"attack_rate":frac})
    return float(ece), pd.DataFrame(rows)

def calibration_metrics(y, p):
    p = np.clip(np.asarray(p, dtype=float), EPS, 1-EPS)
    y = np.asarray(y, dtype=int)
    ece, _ = ece_score(y, p, ECE_BINS)
    return {
        "brier": brier_score_loss(y, p),
        "nll": log_loss(y, np.c_[1-p,p], labels=[0,1]),
        "ece": ece,
        "auroc": roc_auc_score(y, p),
        "average_precision": average_precision_score(y, p),
    }

class PlattOnProbability:
    def fit(self, p, y):
        p = np.clip(np.asarray(p,float), EPS, 1-EPS)
        z = np.log(p/(1-p)).reshape(-1,1)
        self.model = LogisticRegression(solver="lbfgs", random_state=SEED)
        self.model.fit(z, np.asarray(y,int))
        return self
    def predict(self, p):
        p = np.clip(np.asarray(p,float), EPS, 1-EPS)
        z = np.log(p/(1-p)).reshape(-1,1)
        return self.model.predict_proba(z)[:,1]

class IsotonicCalibrator:
    def fit(self, p, y):
        self.model = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
        self.model.fit(np.asarray(p,float), np.asarray(y,int))
        return self
    def predict(self, p):
        return np.asarray(self.model.predict(np.asarray(p,float)), float)

def fit_calibrators(fit_df):
    p = fit_df["y_probability_attack"].to_numpy(float)
    y = fit_df["y_true"].to_numpy(int)
    return {
        "raw": None,
        "platt": PlattOnProbability().fit(p,y),
        "isotonic": IsotonicCalibrator().fit(p,y),
    }

def apply_calibrator(cal, p):
    if cal is None:
        return np.asarray(p,float)
    return np.clip(cal.predict(p),0,1)

In [ ]:
rf_calibrators = fit_calibrators(cal_fit)
rows = []
for method, cal in rf_calibrators.items():
    p_pol = apply_calibrator(cal, policy_val["y_probability_attack"].to_numpy(float))
    rows.append({"method":method, **calibration_metrics(policy_val["y_true"], p_pol)})

rf_cal_selection = pd.DataFrame(rows).sort_values(["brier","ece","nll"]).reset_index(drop=True)
SELECTED_CALIBRATION = rf_cal_selection.iloc[0]["method"]
rf_calibrator = rf_calibrators[SELECTED_CALIBRATION]

print(rf_cal_selection)
print("\nSelected RF calibration method:", SELECTED_CALIBRATION)

rf_test_raw_p = rf_test["y_probability_attack"].to_numpy(float)
rf_test_cal_p = apply_calibrator(rf_calibrator, rf_test_raw_p)
rf_test_y = rf_test["y_true"].to_numpy(int)

calibration_test = pd.DataFrame([
    {"method":"raw", **calibration_metrics(rf_test_y, rf_test_raw_p)},
    {"method":SELECTED_CALIBRATION, **calibration_metrics(rf_test_y, rf_test_cal_p)},
])
# Reported for completeness (the selection itself used validation only).
calibration_reported = pd.DataFrame(
    [{"method": "raw", "selected": SELECTED_CALIBRATION == "raw", **calibration_metrics(rf_test_y, rf_test_raw_p)}] +
    [{"method": name, "selected": SELECTED_CALIBRATION == name,
      **calibration_metrics(rf_test_y, apply_calibrator(cal, rf_test_raw_p))}
     for name, cal in rf_calibrators.items() if name != "raw"])
calibration_reported.to_csv(OUT_DIR/"tables"/"Step7_RF_Calibration_All_Methods_Test.csv", index=False)
base_accuracy_full = float(((rf_test_raw_p >= 0.5).astype(int) == rf_test_y).mean())
calibration_test.to_csv(OUT_DIR/"tables"/"Step7_RF_Calibration_Test.csv", index=False)
calibration_test

In [ ]:
all9_cal_rows = []
if RUN_ALL9_CALIBRATION:
    for model in sorted(p_val_all["model"].unique()):
        v = model_prob(p_val_all, model)
        t = model_prob(p_test_all, model)
        m = len(v)//2
        fit_m, pol_m = v.iloc[:m], v.iloc[m:]
        cals = fit_calibrators(fit_m)

        candidates = []
        for name, cal in cals.items():
            pp = apply_calibrator(cal, pol_m["y_probability_attack"])
            candidates.append({"method":name, **calibration_metrics(pol_m["y_true"], pp)})
        sel = pd.DataFrame(candidates).sort_values(["brier","ece","nll"]).iloc[0]["method"]

        raw_m = calibration_metrics(t["y_true"], t["y_probability_attack"])
        p_cal = apply_calibrator(cals[sel], t["y_probability_attack"])
        cal_m = calibration_metrics(t["y_true"], p_cal)
        all9_cal_rows.append({
            "model":model, "selected_calibration":sel,
            "raw_brier":raw_m["brier"], "calibrated_brier":cal_m["brier"],
            "raw_ece":raw_m["ece"], "calibrated_ece":cal_m["ece"],
            "raw_nll":raw_m["nll"], "calibrated_nll":cal_m["nll"],
            "raw_auroc":raw_m["auroc"], "calibrated_auroc":cal_m["auroc"],
        })

all9_calibration = pd.DataFrame(all9_cal_rows)
all9_calibration.to_csv(OUT_DIR/"tables"/"TableS_Step7_All9_Calibration.csv", index=False)
all9_calibration

In [ ]:
# Reliability bins for Figure 10(a); the figure itself is drawn once the
# selective-prediction table exists, so that calibration and deferral are
# presented together.
_, raw_bins = ece_score(rf_test_y, rf_test_raw_p, ECE_BINS)
_, cal_bins = ece_score(rf_test_y, rf_test_cal_p, ECE_BINS)
print(f"Reliability bins computed (raw: {len(raw_bins)}, calibrated: {len(cal_bins)} occupied bins of {ECE_BINS}).")


## Selective prediction on the entire temporal test set

Thresholds are determined **without test labels** from policy-validation. The test set is used only to measure realized coverage and error capture.

In [ ]:
def binary_entropy(p):
    p = np.clip(np.asarray(p,float), EPS, 1-EPS)
    return -(p*np.log(p) + (1-p)*np.log(1-p))

def uncertainty_confidence(p):
    p = np.asarray(p,float)
    return 1.0 - np.maximum(p,1-p)

def build_wide(prob_df):
    return prob_df.pivot(index="flow_id", columns="model", values="y_probability_attack")

wide_val = build_wide(p_val_all)
wide_test = build_wide(p_test_all)

policy_ids = policy_val["flow_id"].tolist()
test_ids_ordered = rf_test["flow_id"].tolist()

policy_raw = policy_val["y_probability_attack"].to_numpy(float)
policy_cal = apply_calibrator(rf_calibrator, policy_raw)

selective_scores_val = {
    "RF raw uncertainty": uncertainty_confidence(policy_raw),
    "RF calibrated uncertainty": uncertainty_confidence(policy_cal),
    "RF calibrated entropy": binary_entropy(policy_cal),
    "All-9 probability disagreement": wide_val.loc[policy_ids].std(axis=1).to_numpy(float),
}
selective_scores_test = {
    "RF raw uncertainty": uncertainty_confidence(rf_test_raw_p),
    "RF calibrated uncertainty": uncertainty_confidence(rf_test_cal_p),
    "RF calibrated entropy": binary_entropy(rf_test_cal_p),
    "All-9 probability disagreement": wide_test.loc[test_ids_ordered].std(axis=1).to_numpy(float),
}

def threshold_for_budget(scores, budget):
    return float(np.quantile(np.asarray(scores,float), 1-budget))

def selective_metrics(y, p, defer_mask):
    y = np.asarray(y,int); p = np.asarray(p,float); defer_mask = np.asarray(defer_mask,bool)
    pred = (p>=0.5).astype(int)
    keep = ~defer_mask
    base_errors = pred != y
    base_fn = (pred==0) & (y==1)
    out = {
        "n":len(y),
        "deferred":int(defer_mask.sum()),
        "deferral_rate":float(defer_mask.mean()),
        "coverage":float(keep.mean()),
        "deferred_error_capture":float((base_errors & defer_mask).sum()/max(base_errors.sum(),1)),
        "deferred_false_negative_capture":float((base_fn & defer_mask).sum()/max(base_fn.sum(),1)),
    }
    if keep.sum():
        m = classification_metrics(y[keep], p[keep])
        out.update({
            "selective_accuracy":m["accuracy"],
            "selective_balanced_accuracy":m["balanced_accuracy"],
            "selective_macro_f1":m["macro_f1"],
            "selective_mcc":m["mcc"],
            "selective_fnr":m["fnr"],
            "selective_risk":1-m["accuracy"],
        })
    return out

sel_rows = []
for method in selective_scores_val:
    for budget in SELECTIVE_BUDGETS:
        thr = threshold_for_budget(selective_scores_val[method], budget)
        defer = selective_scores_test[method] >= thr
        sel_rows.append({
            "method":method,
            "target_deferral_budget":budget,
            "validation_score_threshold":thr,
            **selective_metrics(rf_test_y, rf_test_raw_p, defer)
        })

selective_table = pd.DataFrame(sel_rows)
selective_table.to_csv(OUT_DIR/"tables"/"Step7_Selective_Prediction_Temporal_Test.csv", index=False)
selective_table.head(12)

In [ ]:
# ---------------------------------------------------------------------------
# Tie-aware, coverage-controlled selective prediction.
# Calibrators such as isotonic regression map many raw probabilities onto the
# same value, so a validation quantile threshold cannot hit an intermediate
# budget: at the nominal 8% budget the calibrated score defers ~49% of the test
# partition because the threshold falls inside a large tie block. The table
# below therefore ALSO reports a coverage-controlled policy that defers exactly
# ceil(budget x n) flows ranked by (score, raw-uncertainty tie-break, flow_id).
# Ranking uses no labels; it controls workload exactly, as in standard
# risk-coverage analysis.
# ---------------------------------------------------------------------------
tie_break = uncertainty_confidence(rf_test_raw_p)
cov_rows = []
for method, scores in selective_scores_test.items():
    order = np.lexsort((np.asarray(test_ids_ordered), -tie_break, -np.asarray(scores, float)))
    for budget in SELECTIVE_BUDGETS:
        k = int(math.ceil(budget * len(order)))
        defer = np.zeros(len(order), bool); defer[order[:k]] = True
        cov_rows.append({"method": method, "target_deferral_budget": budget, "selection": "coverage-controlled",
                         **selective_metrics(rf_test_y, rf_test_raw_p, defer)})
selective_coverage_table = pd.DataFrame(cov_rows)
selective_coverage_table.to_csv(OUT_DIR/"tables"/"Step7_Selective_Prediction_Coverage_Controlled.csv", index=False)

_tie = selective_table.assign(rate_gap=lambda d: (d.deferral_rate - d.target_deferral_budget).abs())
TIE_MAX_GAP = float(_tie["rate_gap"].max())
TIE_WORST = _tie.sort_values("rate_gap", ascending=False).iloc[0]
print(f"Validation-threshold policy: largest gap between target and realised deferral is {100*TIE_MAX_GAP:.1f} pp "
      f"({TIE_WORST['method']} at a {100*TIE_WORST['target_deferral_budget']:.0f}% budget deferred "
      f"{100*TIE_WORST['deferral_rate']:.1f}%), caused by tied calibrated probabilities.")
print("Coverage-controlled policy realises the requested budget exactly and is used for the risk-coverage figures.")
selective_coverage_table.head(8)


In [ ]:
# Figure 10 — calibration and selective prediction (temporal test)
fig, axes = plt.subplots(2, 2, figsize=(DOUBLE_COL, 5.6))
fig.subplots_adjust(hspace=0.48, wspace=0.34)

ax = axes[0, 0]
ax.plot([0, 1], [0, 1], ls="--", lw=0.8, color="#9A9A9A", label="Perfect calibration")
raw_row = calibration_test[calibration_test.method == "raw"].iloc[0]
sel_row = calibration_test[calibration_test.method == SELECTED_CALIBRATION].iloc[0]
ax.plot(raw_bins["mean_probability"], raw_bins["attack_rate"], marker="o", ms=4, color=OUTCOME_COLORS["raw"],
        label=f"Raw RF (ECE {raw_row.ece:.4f})")
ax.plot(cal_bins["mean_probability"], cal_bins["attack_rate"], marker="s", ms=4,
        color=OUTCOME_COLORS.get(SELECTED_CALIBRATION, "#0072B2"),
        label=f"RF + {SELECTED_CALIBRATION} (ECE {sel_row.ece:.4f})")
ax.set_xlabel("Mean predicted attack probability"); ax.set_ylabel("Observed attack frequency")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc="upper left")
ygrid(ax, "both"); add_panel_label(ax, "a")

ax = axes[0, 1]
for method, g in selective_coverage_table[selective_coverage_table.method != "RF calibrated entropy"].groupby("method"):
    g = g.sort_values("coverage")
    ax.plot(100*g["coverage"], 100*g["selective_risk"], marker=SIGNAL_MARKERS.get(method, "o"), ms=4,
            color=SIGNAL_COLORS.get(method, NEUTRAL), label=method,
            ls="--" if method == "RF calibrated entropy" else "-", alpha=0.95 if method == "RF calibrated entropy" else 1.0)
base_risk = 100*(1 - base_accuracy_full)
ax.axhline(base_risk, ls="--", lw=0.8, color="#9A9A9A")
ax.text(100.5, base_risk, f"no deferral ({base_risk:.2f}%)", fontsize=6.8, color=NEUTRAL, va="center", ha="right")
ax.set_xlabel("Coverage (% of decisions retained)"); ax.set_ylabel("Selective risk (%)")
ax.invert_xaxis(); ygrid(ax, "both"); add_panel_label(ax, "b")

ax = axes[1, 0]
for method, g in selective_coverage_table[selective_coverage_table.method != "RF calibrated entropy"].groupby("method"):
    g = g.sort_values("deferral_rate")
    ax.plot(100*g["deferral_rate"], 100*g["deferred_error_capture"], marker=SIGNAL_MARKERS.get(method, "o"),
            ms=4, color=SIGNAL_COLORS.get(method, NEUTRAL), label=method,
            ls="--" if method == "RF calibrated entropy" else "-")
b = np.array(SELECTIVE_BUDGETS)
ax.plot(100*b, 100*b, ls=":", lw=0.9, color="#9A9A9A", label="Random deferral")
ax.set_xlabel("Realised deferral rate (% of decisions)"); ax.set_ylabel("Model errors captured (%)")
ax.set_ylim(0, 100); ygrid(ax, "both"); add_panel_label(ax, "c")
handles = [Line2D([], [], color=SIGNAL_COLORS.get(m, NEUTRAL), marker=SIGNAL_MARKERS.get(m, "o"), ms=4, label=m)
           for m in selective_coverage_table["method"].unique() if m != "RF calibrated entropy"] + [Line2D([], [], color="#9A9A9A", ls=":", label="Random deferral")]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=2, fontsize=7)

ax = axes[1, 1]
cal_all = calibration_reported.set_index("method")
methods = [m for m in ["raw", "platt", "isotonic"] if m in cal_all.index]
ypos = np.arange(len(methods))[::-1]
for metric, mk, lbl in (("brier", "o", "Brier"), ("ece", "s", "ECE")):
    for y_, m in zip(ypos, methods):
        ax.plot(cal_all.at[m, metric], y_ + (0.16 if metric == "brier" else -0.16), mk,
                color=OUTCOME_COLORS.get(m, NEUTRAL), ms=5, mec="white", mew=0.4)
ax.set_yticks(ypos, [f"{m}{'  (selected)' if m == SELECTED_CALIBRATION else ''}" for m in methods], fontsize=7.5)
ax.set_xscale("log"); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlabel("Value (log scale); lower is better"); ax.tick_params(axis="y", length=0)
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=NEUTRAL, label="Brier"),
                   Line2D([], [], marker="s", ls="", color=NEUTRAL, label="ECE")],
          loc="upper center", bbox_to_anchor=(0.5, -0.30), ncol=2)
ygrid(ax, "x"); add_panel_label(ax, "d", x=-0.30)

CAPTION_FIG10 = (
    f"Figure 10. Probability quality and selective prediction for the frozen Step-6 reference classifier (Random Forest) on the "
    f"temporal test partition (n = {len(rf_test_y):,}). (a) Reliability curves before and after the validation-selected calibration, with "
    f"expected calibration error (ECE, {ECE_BINS} bins). (b) Risk–coverage trade-off for four uncertainty signals under coverage-controlled "
    f"deferral; the dashed line is the risk without deferral. (c) Share of the classifier's errors captured against the realised "
    f"deferral rate; the dotted line is the expectation under random deferral. Calibrated entropy is a monotone transform of calibrated "
    f"uncertainty for binary decisions, selects identical flows and is therefore reported in the tables only. (d) Brier score and ECE for raw and calibrated probabilities (logarithmic axis); the "
    f"calibration method was selected on held-out validation data, never on the test partition.")
save_publication_figure(fig, "Figure10_Calibration_Selective_Prediction", CAPTION_FIG10, 4)
plt.show(); plt.close(fig); print(CAPTION_FIG10)


## Human–AI collaboration on the canonical oversight pool

The final system is evaluated over the **entire temporal test set**. Non-deferred cases keep the frozen RF decision; deferred eligible cases may receive a configured-profile decision. Workload is reported as a fraction of the full temporal test partition.

In [ ]:
rf_val_map = rf_val.set_index("flow_id")[["y_true","y_probability_attack"]].rename(columns={"y_probability_attack":"rf_probability"})
rf_test_map = rf_test.set_index("flow_id")[["y_true","y_probability_attack"]].rename(columns={"y_probability_attack":"rf_probability"})

ov_val = oversight[oversight["temporal_split_step7"]=="validation"].copy().set_index("flow_id").join(rf_val_map,how="inner",rsuffix="_step6")
ov_test = oversight[oversight["temporal_split_step7"]=="test"].copy().set_index("flow_id").join(rf_test_map,how="inner",rsuffix="_step6")

ov_val["rf_pred"] = (ov_val["rf_probability"]>=.5).astype(int)
ov_test["rf_pred"] = (ov_test["rf_probability"]>=.5).astype(int)
ov_val["rf_cal_probability"] = apply_calibrator(rf_calibrator, ov_val["rf_probability"])
ov_test["rf_cal_probability"] = apply_calibrator(rf_calibrator, ov_test["rf_probability"])
ov_val["all9_disagreement"] = wide_val.loc[ov_val.index].std(axis=1)
ov_test["all9_disagreement"] = wide_test.loc[ov_test.index].std(axis=1)

print("Validation eligible pool:", len(ov_val), f"({len(ov_val)/len(rf_val):.2%})")
print("Test eligible pool:", len(ov_test), f"({len(ov_test)/len(rf_test):.2%})")

## Oversight reachability and the ambiguity/uncertainty distinction

Profile override can only act on flows that are **review-eligible**, i.e. inside the generator's Top-K ambiguity subset fixed when the dataset was built. That subset was defined from the benchmark-construction score, not from the trained classifier's uncertainty, so the two need not coincide. The next two cells quantify (i) how many of the classifier's errors are reachable at all, giving a hard ceiling for every override policy, and (ii) how closely generator ambiguity tracks model uncertainty. Both are properties of the frozen benchmark and involve no tuning.

In [ ]:
# ---------------------------------------------------------------------------
# Oversight reachability: how much of the model's error can profile override
# reach at all? The review-eligible pool is the generator's Top-K AMBIGUITY
# subset (fixed when the dataset was built from the construction score), which
# is not the same thing as the trained model's uncertainty. Everything here is
# a property of the frozen benchmark; it uses no tuning.
# ---------------------------------------------------------------------------
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z**2/(4*n*n)) / d
    return (max(0.0, c-h), min(1.0, c+h))

test_index = full_test.index if "full_test" in dir() else rf_test.set_index("flow_id").index
_rf_test_idx = rf_test.set_index("flow_id")
in_pool = _rf_test_idx.index.isin(ov_test.index)
y_t = _rf_test_idx["y_true"].astype(int).to_numpy()
pred_t = (_rf_test_idx["y_probability_attack"].to_numpy(float) >= 0.5).astype(int)
err_t = pred_t != y_t
fn_t = (pred_t == 0) & (y_t == 1)

pool_ids = _rf_test_idx.index[in_pool]
prof_pool = ov_test.loc[pool_ids, "profile_decision"].astype(int).to_numpy()
y_pool = ov_test.loc[pool_ids, "is_attack"].astype(int).to_numpy()
ai_pool = pred_t[in_pool]
correctable = int(((ai_pool != y_pool) & (prof_pool == y_pool)).sum())
harm_exposure = int(((ai_pool == y_pool) & (prof_pool != y_pool)).sum())

N_TEST = len(y_t)
CEILING_RATE = correctable / N_TEST
reach_rows = [
    {"quantity": "temporal test decisions", "count": N_TEST, "percent_of_test": 100.0},
    {"quantity": "review-eligible pool (generator Top-K ambiguity)", "count": int(in_pool.sum()),
     "percent_of_test": 100*in_pool.mean()},
    {"quantity": "model errors (all)", "count": int(err_t.sum()), "percent_of_test": 100*err_t.mean()},
    {"quantity": "model false negatives (all)", "count": int(fn_t.sum()), "percent_of_test": 100*fn_t.mean()},
    {"quantity": "model errors inside eligible pool", "count": int((err_t & in_pool).sum()),
     "percent_of_test": 100*(err_t & in_pool).mean()},
    {"quantity": "correctable (model wrong, profile right)", "count": correctable,
     "percent_of_test": 100*CEILING_RATE},
    {"quantity": "harm exposure (model right, profile wrong)", "count": harm_exposure,
     "percent_of_test": 100*harm_exposure/N_TEST},
]
reachability_table = pd.DataFrame(reach_rows)
lo, hi = wilson_ci(correctable, N_TEST)
reachability_table["ci95_lower_percent"] = [100*wilson_ci(int(r), N_TEST)[0] for r in reachability_table["count"]]
reachability_table["ci95_upper_percent"] = [100*wilson_ci(int(r), N_TEST)[1] for r in reachability_table["count"]]
reachability_table.to_csv(OUT_DIR/"tables"/"Step7_Oversight_Reachability.csv", index=False)

ERROR_COVERAGE = float((err_t & in_pool).sum() / max(err_t.sum(), 1))
POOL_AI_ACC = float((ai_pool == y_pool).mean())
POOL_PROFILE_ACC = float((prof_pool == y_pool).mean())
print(f"Eligible pool: {in_pool.sum():,} of {N_TEST:,} test decisions ({100*in_pool.mean():.2f}%)")
print(f"Model errors inside the pool: {(err_t & in_pool).sum()} of {err_t.sum()} ({100*ERROR_COVERAGE:.1f}% of all errors)")
print(f"Inside the pool — model accuracy {POOL_AI_ACC:.3f}, configured-profile accuracy {POOL_PROFILE_ACC:.3f}")
print(f"Correctable cases: {correctable} (oracle ceiling {100*CEILING_RATE:.3f}% of decisions, "
      f"95% CI {100*lo:.3f}-{100*hi:.3f}%); harm exposure: {harm_exposure}")
print("\nInterpretation: any profile-override policy is bounded by the correctable count; the ceiling is a property "
      "of the frozen benchmark (ambiguity-selected pool vs model uncertainty), not of the routing signal.")
reachability_table


In [ ]:
# ---------------------------------------------------------------------------
# Generator ambiguity vs model uncertainty: are they the same notion?
# ---------------------------------------------------------------------------
amb_map = df.set_index("flow_id")["ambiguity_score"]
test_amb = amb_map.loc[_rf_test_idx.index].to_numpy(float)
test_unc = uncertainty_confidence(_rf_test_idx["y_probability_attack"].to_numpy(float))
auc_pool = roc_auc_score(in_pool.astype(int), test_unc)
rho = float(pd.Series(test_amb).corr(pd.Series(test_unc), method="spearman"))
overlap_table = pd.DataFrame([{
    "test_rows": N_TEST,
    "eligible_pool_rate": float(in_pool.mean()),
    "auroc_model_uncertainty_predicts_pool_membership": float(auc_pool),
    "spearman_ambiguity_vs_model_uncertainty": rho,
    "model_errors_inside_pool_rate": ERROR_COVERAGE,
    "model_accuracy_inside_pool": POOL_AI_ACC,
    "profile_accuracy_inside_pool": POOL_PROFILE_ACC,
    "top8pct_model_uncertainty_overlap_with_pool": float(
        np.mean(in_pool[np.argsort(-test_unc)[:int(round(0.08*N_TEST))]]))
}])
overlap_table.to_csv(OUT_DIR/"tables"/"Step7_Ambiguity_vs_Model_Uncertainty.csv", index=False)
OVERLAP_TOP8 = float(overlap_table.iloc[0]["top8pct_model_uncertainty_overlap_with_pool"])
print(f"Model uncertainty predicts pool membership with AUROC {auc_pool:.3f}; "
      f"Spearman(ambiguity, model uncertainty) = {rho:.3f}")
print(f"Of the 8% most model-uncertain test flows, {100*OVERLAP_TOP8:.1f}% are in the generator's ambiguity pool.")
overlap_table.T


In [ ]:
policy_ov_ids = set(policy_val["flow_id"])
ov_policy = ov_val.loc[ov_val.index.intersection(policy_ov_ids)].copy()

gate_rows = []
for t in np.round(np.arange(0.50, 0.951, 0.05), 2):
    use_profile = (ov_policy["profile_confidence"]>=t) & (ov_policy["profile_decision"].astype(int)!=ov_policy["rf_pred"])
    final = ov_policy["rf_pred"].to_numpy().copy()
    final[use_profile.to_numpy()] = ov_policy.loc[use_profile,"profile_decision"].astype(int).to_numpy()
    y = ov_policy["is_attack"].astype(int).to_numpy()
    ai = ov_policy["rf_pred"].to_numpy()
    corrections = int(((ai!=y)&(final==y)).sum())
    harms = int(((ai==y)&(final!=y)).sum())
    gate_rows.append({
        "profile_confidence_threshold":float(t),
        "corrections":corrections,
        "harms":harms,
        "net_corrections":corrections-harms,
        "macro_f1":f1_score(y,final,average="macro",zero_division=0)
    })

gate_selection = pd.DataFrame(gate_rows).sort_values(
    ["net_corrections","macro_f1","profile_confidence_threshold"],
    ascending=[False,False,False]
).reset_index(drop=True)
PROFILE_GATE = float(gate_selection.iloc[0]["profile_confidence_threshold"])
gate_selection.to_csv(OUT_DIR/"tables"/"Step7_Profile_Confidence_Gate_Validation.csv",index=False)
print("Validation-selected profile confidence gate:", PROFILE_GATE)
_top = gate_selection.iloc[0]
print(f"At the selected gate: {int(_top['corrections'])} corrections vs {int(_top['harms'])} harms on the policy-validation "
      f"pool (net {int(_top['net_corrections'])}). A gate of 0.95 with zero corrections and zero harms means validation "
      f"selected 'effectively never override' as the best available policy.")
gate_selection.head()

In [ ]:
full_test = rf_test.set_index("flow_id").copy()
full_test["rf_pred"] = (full_test["y_probability_attack"]>=.5).astype(int)
full_test_y = full_test["y_true"].astype(int).to_numpy()
full_test_ai = full_test["rf_pred"].astype(int).to_numpy()

def full_system_metrics(final_pred, deferred_ids=None):
    final_pred = np.asarray(final_pred,int)
    tn,fp,fn,tp = confusion_matrix(full_test_y,final_pred,labels=[0,1]).ravel()
    out = {
        "accuracy":accuracy_score(full_test_y,final_pred),
        "balanced_accuracy":balanced_accuracy_score(full_test_y,final_pred),
        "macro_f1":f1_score(full_test_y,final_pred,average="macro",zero_division=0),
        "mcc":matthews_corrcoef(full_test_y,final_pred) if len(np.unique(final_pred))>1 else 0.0,
        "fpr":fp/(fp+tn) if fp+tn else np.nan,
        "fnr":fn/(fn+tp) if fn+tp else np.nan,
    }
    if deferred_ids is not None:
        mask = full_test.index.isin(set(deferred_ids))
        out["deferred"] = int(mask.sum())
        out["workload_rate_full_test"] = float(mask.mean())
        out["ai_errors_corrected"] = int(((full_test_ai!=full_test_y)&(final_pred==full_test_y)&mask).sum())
        out["ai_correct_harmed"] = int(((full_test_ai==full_test_y)&(final_pred!=full_test_y)&mask).sum())
        out["net_errors_avoided"] = out["ai_errors_corrected"] - out["ai_correct_harmed"]
        out["attack_false_negatives_corrected"] = int(((full_test_ai==0)&(full_test_y==1)&(final_pred==1)&mask).sum())
    return out

def apply_profile_override(selected_ids, confidence_gate=None, oracle=False):
    final = full_test_ai.copy()
    pos = {fid:i for i,fid in enumerate(full_test.index)}
    for fid in selected_ids:
        if fid not in ov_test.index or fid not in pos:
            continue
        i = pos[fid]
        y = int(ov_test.at[fid,"is_attack"])
        ai = int(final[i])
        prof = int(ov_test.at[fid,"profile_decision"])
        if oracle:
            if ai!=y and prof==y:
                final[i]=prof
            continue
        if confidence_gate is None:
            final[i]=prof
        else:
            conf = float(ov_test.at[fid,"profile_confidence"])
            if prof!=ai and conf>=confidence_gate:
                final[i]=prof
    return final

In [ ]:
routing_val = {
    "RF raw uncertainty": uncertainty_confidence(ov_val["rf_probability"]),
    "RF calibrated uncertainty": uncertainty_confidence(ov_val["rf_cal_probability"]),
    "All-9 disagreement": ov_val["all9_disagreement"].to_numpy(float),
}
routing_test = {
    "RF raw uncertainty": uncertainty_confidence(ov_test["rf_probability"]),
    "RF calibrated uncertainty": uncertainty_confidence(ov_test["rf_cal_probability"]),
    "All-9 disagreement": ov_test["all9_disagreement"].to_numpy(float),
}

collab_rows = [{
    "routing_method":"AI-only","override_policy":"none","target_full_test_budget":0.0,
    "validation_threshold":np.nan, **full_system_metrics(full_test_ai),
    "generated_latency_total_s":0.0,"generated_latency_mean_s":0.0
}]

rng = np.random.default_rng(SEED)

for budget in OVERSIGHT_BUDGETS_FULL_TEST:
    eligible_rate_val = len(ov_val)/len(rf_val)
    eligible_fraction = min(1.0,budget/eligible_rate_val)

    for method in routing_val:
        thr = threshold_for_budget(routing_val[method],eligible_fraction)
        selected = ov_test.index[np.asarray(routing_test[method])>=thr].tolist()

        for policy,gate,oracle in [
            ("unconditional profile override",None,False),
            ("confidence-gated profile override",PROFILE_GATE,False),
            ("oracle upper bound",None,True),
        ]:
            final = apply_profile_override(selected,confidence_gate=gate,oracle=oracle)
            m = full_system_metrics(final,selected)
            lat = ov_test.loc[ov_test.index.intersection(selected),"generated_decision_latency"]
            collab_rows.append({
                "routing_method":method,"override_policy":policy,
                "target_full_test_budget":budget,"validation_threshold":thr,
                **m,
                "generated_latency_total_s":float(lat.sum()) if len(lat) else 0.0,
                "generated_latency_mean_s":float(lat.mean()) if len(lat) else 0.0,
            })

    n_target = min(len(ov_test),int(round(budget*len(full_test))))
    rand_records = []
    for rep in range(RANDOM_ROUTING_REPS):
        ids = rng.choice(ov_test.index.to_numpy(),size=n_target,replace=False).tolist()
        final = apply_profile_override(ids,confidence_gate=PROFILE_GATE,oracle=False)
        rand_records.append(full_system_metrics(final,ids))
    rr = pd.DataFrame(rand_records)
    collab_rows.append({
        "routing_method":"Matched-budget random routing",
        "override_policy":"confidence-gated profile override",
        "target_full_test_budget":budget,
        "validation_threshold":np.nan,
        **{c: float(rr[c].mean()) for c in rr.columns},
        **{f"{c}_sd": float(rr[c].std(ddof=1)) for c in ["net_errors_avoided", "ai_errors_corrected", "ai_correct_harmed"]},
        "generated_latency_total_s":np.nan,
        "generated_latency_mean_s":np.nan,
    })

collaboration_table = pd.DataFrame(collab_rows)
collaboration_table.to_csv(OUT_DIR/"tables"/"Step7_HumanAI_Oversight_Mechanisms.csv",index=False)
collaboration_table.head(16)

## Learned deferral utility baseline

To avoid in-sample RF training probabilities, the notebook performs one additional fit of the frozen RF configuration with `oob_score=True` and uses out-of-bag probabilities to construct the temporal-training defer labels.

The learned routing utility is:

\[
U(x)=P(\mathrm{correction}\mid x)-P(\mathrm{harm}\mid x).
\]

This is a transparent learned rejector baseline rather than an implementation of the exact Mozannar–Sontag surrogate loss.

In [ ]:
learned_defer_table = pd.DataFrame()

if RUN_LEARNED_DEFER:
    rf_frozen = joblib.load(rf_model_path)

    train_df = df[df["temporal_split_step7"]=="train"].copy().reset_index(drop=True)
    feature_list = rf_meta["features"]
    X_train = train_df[feature_list].copy()
    y_train = train_df["is_attack"].astype(int).to_numpy()

    rf_oob = clone(rf_frozen)
    rf_oob.set_params(clf__oob_score=True)
    t0 = time.perf_counter()
    rf_oob.fit(X_train,y_train)
    oob_fit_seconds = time.perf_counter()-t0
    oob_p = rf_oob.named_steps["clf"].oob_decision_function_[:,1]
    assert np.isfinite(oob_p).all()

    train_prob = pd.DataFrame({
        "flow_id":train_df["flow_id"],
        "ai_probability_oob":oob_p,
        "ai_pred_oob":(oob_p>=.5).astype(int),
    }).set_index("flow_id")

    ov_train = oversight[oversight["temporal_split_step7"]=="train"].copy().set_index("flow_id").join(train_prob,how="inner")
    ov_train["ai_uncertainty"] = uncertainty_confidence(ov_train["ai_probability_oob"])
    ov_val["ai_uncertainty"] = uncertainty_confidence(ov_val["rf_probability"])
    ov_test["ai_uncertainty"] = uncertainty_confidence(ov_test["rf_probability"])

    ytr = ov_train["is_attack"].astype(int)
    ai_tr = ov_train["ai_pred_oob"].astype(int)
    pr_tr = ov_train["profile_decision"].astype(int)
    ov_train["correction_target"] = ((ai_tr!=ytr)&(pr_tr==ytr)).astype(int)
    ov_train["harm_target"] = ((ai_tr==ytr)&(pr_tr!=ytr)).astype(int)

    print("OOB RF fit seconds:",round(oob_fit_seconds,2))
    print("Training correction opportunities:",int(ov_train["correction_target"].sum()))
    print("Training harm opportunities:",int(ov_train["harm_target"].sum()))

    learn_features = feature_list + ["ai_probability_for_defer","ai_uncertainty","oversight_profile"]

    def prep(ov,p_col):
        z = ov.copy()
        z["ai_probability_for_defer"] = z[p_col].astype(float)
        return z[learn_features]

    Xd_tr = prep(ov_train,"ai_probability_oob")
    Xd_val = prep(ov_val,"rf_probability")
    Xd_test = prep(ov_test,"rf_probability")

    cat_cols = [c for c in learn_features if not pd.api.types.is_numeric_dtype(Xd_tr[c])]
    num_cols = [c for c in learn_features if c not in cat_cols]

    defer_pre = ColumnTransformer([
        ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num_cols),
        ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat_cols),
    ])

    def make_defer_classifier():
        return Pipeline([
            ("pre",clone(defer_pre)),
            ("clf",LogisticRegression(C=1.0,class_weight="balanced",max_iter=3000,solver="liblinear",random_state=SEED))
        ])

    corr_model = make_defer_classifier().fit(Xd_tr,ov_train["correction_target"])
    harm_model = make_defer_classifier().fit(Xd_tr,ov_train["harm_target"])

    utility_val = corr_model.predict_proba(Xd_val)[:,1] - harm_model.predict_proba(Xd_val)[:,1]
    corr_test = corr_model.predict_proba(Xd_test)[:,1]
    harm_test = harm_model.predict_proba(Xd_test)[:,1]
    utility_test = corr_test - harm_test

    joblib.dump(corr_model,OUT_DIR/"models"/"learned_defer_correction_model.joblib")
    joblib.dump(harm_model,OUT_DIR/"models"/"learned_defer_harm_model.joblib")
    pd.DataFrame({
        "flow_id":ov_test.index,
        "predicted_correction_probability":corr_test,
        "predicted_harm_probability":harm_test,
        "learned_deferral_utility":utility_test,
    }).to_csv(OUT_DIR/"probabilities"/"learned_deferral_test_scores.csv",index=False)

    learned_rows = []
    eligible_rate_val = len(ov_val)/len(rf_val)
    for budget in OVERSIGHT_BUDGETS_FULL_TEST:
        eligible_fraction = min(1.0,budget/eligible_rate_val)
        thr = threshold_for_budget(utility_val,eligible_fraction)
        selected = ov_test.index[utility_test>=thr].tolist()
        for policy,gate,oracle in [
            ("unconditional profile override",None,False),
            ("confidence-gated profile override",PROFILE_GATE,False),
            ("oracle upper bound",None,True),
        ]:
            final = apply_profile_override(selected,confidence_gate=gate,oracle=oracle)
            learned_rows.append({
                "routing_method":"Learned deferral utility",
                "override_policy":policy,
                "target_full_test_budget":budget,
                "validation_threshold":thr,
                **full_system_metrics(final,selected)
            })

    learned_defer_table = pd.DataFrame(learned_rows)
    learned_defer_table.to_csv(OUT_DIR/"tables"/"Step7_Learned_Deferral.csv",index=False)
    collaboration_table = pd.concat([collaboration_table,learned_defer_table],ignore_index=True)
    collaboration_table.to_csv(OUT_DIR/"tables"/"Step7_HumanAI_Oversight_Mechanisms.csv",index=False)

learned_defer_table.head() if len(learned_defer_table) else "Learned defer disabled"

In [ ]:
# Figure 11 — human-AI oversight: reachability ceiling and realised benefit
fig = plt.figure(figsize=(DOUBLE_COL, 6.0))
gs = fig.add_gridspec(2, 2, hspace=0.62, wspace=0.42)

ax = fig.add_subplot(gs[0, 0])
funnel = reachability_table.iloc[[0, 1, 2, 4, 5, 6]].copy()
labels = ["Test decisions", "Review-eligible pool", "Model errors", "Errors in pool", "Correctable", "Harm exposure"]
cols = ["#B8C6D1", "#7FA3BF", "#D55E00", "#E39B72", "#009E73", "#CC79A7"]
ypos = np.arange(len(funnel))[::-1]
ax.barh(ypos, funnel["count"].to_numpy(), color=cols, height=0.62, edgecolor="white")
for y_, c in zip(ypos, funnel["count"].to_numpy()):
    ax.text(c*1.25, y_, f"{int(c):,}", va="center", fontsize=7, color=NEUTRAL)
ax.set_yticks(ypos, labels, fontsize=7.5); ax.tick_params(axis="y", length=0)
ax.set_xscale("log"); ax.set_xlim(1, funnel["count"].max()*6)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set_xlabel("Flows (log scale)"); ygrid(ax, "x"); add_panel_label(ax, "a", x=-0.52)

ax = fig.add_subplot(gs[0, 1])
gated = collaboration_table[(collaboration_table.override_policy == "confidence-gated profile override")
                            & (collaboration_table.target_full_test_budget > 0)]
for method, g in gated[gated.routing_method != "Matched-budget random routing"].groupby("routing_method"):
    g = g.sort_values("workload_rate_full_test")
    ax.plot(100*g["workload_rate_full_test"], g["net_errors_avoided"], marker=SIGNAL_MARKERS.get(method, "o"), ms=4,
            color=SIGNAL_COLORS.get(method, NEUTRAL), label=method)
rnd = gated[gated.routing_method == "Matched-budget random routing"].sort_values("workload_rate_full_test")
if len(rnd):
    m_, s_ = rnd["net_errors_avoided"].to_numpy(), rnd.get("net_errors_avoided_sd", pd.Series(np.zeros(len(rnd)))).to_numpy()
    ax.plot(100*rnd["workload_rate_full_test"], m_, ls="--", color=POLICY_COLORS["Matched-budget random routing"],
            label=f"Random routing (mean of {RANDOM_ROUTING_REPS})")
    ax.fill_between(100*rnd["workload_rate_full_test"], m_-s_, m_+s_, color="#BBBBBB", alpha=0.35, lw=0)
ax.axhline(0, lw=0.8, color="#9A9A9A")
ax.axhline(reachability_table.iloc[5]["count"], ls=":", lw=1.0, color="#009E73")
ax.text(0.02, reachability_table.iloc[5]["count"], " reachability ceiling", fontsize=6.8, color="#009E73",
        va="bottom", ha="left", transform=ax.get_yaxis_transform())
ax.set_xlabel("Review workload (% of temporal test)"); ax.set_ylabel("Net errors avoided")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=6.6)
ygrid(ax); add_panel_label(ax, "b", x=-0.22)

ax = fig.add_subplot(gs[1, 0])
trade = collaboration_table[(collaboration_table.override_policy.isin(
    ["unconditional profile override", "confidence-gated profile override"]))
    & (collaboration_table.target_full_test_budget > 0)]
for policy, g in trade.groupby("override_policy"):
    ax.scatter(g["ai_correct_harmed"], g["ai_errors_corrected"], s=26, alpha=0.85,
               color=POLICY_COLORS[policy], edgecolors="white", linewidths=0.4, label=policy)
mx = max(1.0, float(trade["ai_correct_harmed"].max()))
ax.plot([0, mx], [0, mx], ls="--", lw=0.8, color="#9A9A9A")
ax.text(mx, mx*0.86, "break-even ", fontsize=6.6, color=NEUTRAL, va="center", ha="right")
ax.set_xlabel("Correct model decisions harmed"); ax.set_ylabel("Model errors corrected")
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.0), fontsize=6.6); ygrid(ax, "both"); add_panel_label(ax, "c")

ax = fig.add_subplot(gs[1, 1])
rows = []
for policy in ["unconditional profile override", "confidence-gated profile override", "oracle upper bound"]:
    g = collaboration_table[(collaboration_table.override_policy == policy)
                            & (collaboration_table.target_full_test_budget > 0)]
    if len(g):
        best = g.sort_values("net_errors_avoided", ascending=False).iloc[0]
        rows.append((policy, best["ai_errors_corrected"], best["ai_correct_harmed"]))
ypos = np.arange(len(rows))[::-1]
for y_, (policy, corr, harm) in zip(ypos, rows):
    for val, off, mk, lbl in ((corr, 0.16, "o", "corrected"), (harm, -0.16, "X", "harmed")):
        lo_, hi_ = wilson_ci(int(val), N_TEST)
        ax.errorbar(100*val/N_TEST, y_+off, xerr=[[100*val/N_TEST - 100*lo_], [100*hi_ - 100*val/N_TEST]],
                    fmt=mk, ms=5, color=POLICY_COLORS[policy], elinewidth=0.9, capsize=1.6, mec="white", mew=0.4)
        ax.text(100*hi_ + 0.004, y_+off, f"{int(val)}", fontsize=6.6, color=NEUTRAL, va="center")
ax.set_yticks(ypos, [p.replace(" profile override", "").replace(" upper bound", "") for p, _, _ in rows], fontsize=7.5)
ax.tick_params(axis="y", length=0)
ax.set_xlabel("% of temporal test decisions (95% Wilson CI)")
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=NEUTRAL, label="Errors corrected"),
                   Line2D([], [], marker="X", ls="", color=NEUTRAL, label="Correct decisions harmed")],
          loc="lower right", fontsize=6.8)
ygrid(ax, "x"); add_panel_label(ax, "d", x=-0.38)

CAPTION_FIG11 = (
    f"Figure 11. Human–AI oversight on the temporal test partition (n = {N_TEST:,}); configured oversight profiles, not human "
    f"analysts. (a) Reachability of profile override (logarithmic axis): the review-eligible pool is the generator's Top-K "
    f"ambiguity subset, and only {int(reachability_table.iloc[4]['count'])} of {int(reachability_table.iloc[2]['count'])} model "
    f"errors fall inside it, of which {int(reachability_table.iloc[5]['count'])} are correctable by the configured profile, while "
    f"{int(reachability_table.iloc[6]['count'])} correct decisions are exposed to harm. (b) Net errors avoided against realised "
    f"review workload for uncertainty-based routing and matched-budget random routing (mean ± SD over {RANDOM_ROUTING_REPS} "
    f"repetitions); the dotted line is the reachability ceiling from panel (a). (c) Corrections against harms for both override "
    f"policies; the dashed line is break-even. (d) Best achieved corrections and harms per policy as a share of all decisions, "
    f"with 95% Wilson confidence intervals.")
save_publication_figure(fig, "Figure11_Oversight_Reachability_Benefit", CAPTION_FIG11, 4)
plt.show(); plt.close(fig); print(CAPTION_FIG11)


In [ ]:
# Figure 12 — generator ambiguity versus trained-model uncertainty
fig, axes = plt.subplots(1, 3, figsize=(DOUBLE_COL, 2.75))
fig.subplots_adjust(wspace=0.62, bottom=0.30)

ax = axes[0]
for mask, lbl, col, ls in ((in_pool, "In eligible pool", "#0072B2", "-"),
                           (~in_pool, "Outside pool", "#8C8C8C", "--")):
    x = np.sort(test_unc[mask]); y = np.arange(1, x.size+1)/x.size
    ax.step(x, y, where="post", color=col, ls=ls, lw=1.3, label=f"{lbl} (n={mask.sum():,})")
ax.set_xlabel("Model uncertainty  1 − max(p, 1−p)"); ax.set_ylabel("Cumulative proportion")
ax.set_xlim(0, 0.5); ax.set_ylim(0, 1.02); ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.text(0.97, 0.52, f"AUROC {auc_pool:.3f}", transform=ax.transAxes, ha="right", fontsize=7, color=NEUTRAL)
ax.legend(loc="lower right", fontsize=6.8); ygrid(ax, "both"); add_panel_label(ax, "a", x=-0.30)

ax = axes[1]
hb = ax.hexbin(test_amb, test_unc, gridsize=34, bins="log", cmap="Blues", mincnt=1, linewidths=0)
cb = fig.colorbar(hb, ax=ax, fraction=0.045, pad=0.02); cb.set_label("Flows (log)", fontsize=7, labelpad=1)
cb.outline.set_visible(False); cb.ax.tick_params(labelsize=6.5, length=2)
ax.set_xlabel("Generator ambiguity score"); ax.set_ylabel("Model uncertainty")
ax.text(0.03, 0.94, f"Spearman ρ = {rho:.3f}", transform=ax.transAxes, fontsize=7, color=NEUTRAL, va="top")
add_panel_label(ax, "b", x=-0.30)

ax = axes[2]
cm = np.array([[int(((~err_t) & in_pool).sum()), int((err_t & in_pool).sum())],
               [int(((~err_t) & ~in_pool).sum()), int((err_t & ~in_pool).sum())]], float)
ax.imshow(cm / cm.sum(), cmap="Blues", vmin=0, vmax=1, aspect="auto")
for i in range(2):
    for j in range(2):
        ax.text(j, i-0.10, f"{int(cm[i, j]):,}", ha="center", va="center", fontsize=8, fontweight="bold",
                color="white" if cm[i, j]/cm.sum() > 0.5 else "#1A1A1A")
        ax.text(j, i+0.20, f"{100*cm[i, j]/cm.sum():.2f}%", ha="center", va="center", fontsize=6.6,
                color="white" if cm[i, j]/cm.sum() > 0.5 else NEUTRAL)
ax.set_xticks([0, 1], ["Model correct", "Model error"], fontsize=7)
ax.set_yticks([0, 1], ["In pool", "Outside pool"], fontsize=7)
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
add_panel_label(ax, "c", x=-0.34)

CAPTION_FIG12 = (
    f"Figure 12. The generator's ambiguity-selected review pool and the trained classifier's uncertainty are largely distinct "
    f"notions on the temporal test partition (n = {N_TEST:,}). (a) Distribution of model uncertainty inside and outside the "
    f"eligible pool; model uncertainty identifies pool membership with AUROC {auc_pool:.3f}. (b) Joint density of the generator "
    f"ambiguity score and model uncertainty (logarithmic colour scale; Spearman ρ = {rho:.3f}). (c) Cross-tabulation of pool "
    f"membership against classifier correctness, with counts and percentages of all test decisions.")
save_publication_figure(fig, "Figure12_Ambiguity_vs_Model_Uncertainty", CAPTION_FIG12, 3)
plt.show(); plt.close(fig); print(CAPTION_FIG12)


In [ ]:
# Figure quality control (machine-checkable properties only)
REQUIRED_FIGS = {"Figure10_Calibration_Selective_Prediction": 4,
                 "Figure11_Oversight_Reachability_Benefit": 4,
                 "Figure12_Ambiguity_vs_Model_Uncertainty": 3}
BANNED = [r"ml_confidence", r"human confidence", r"real analyst", r"human annotat", r"no (temporal )?drift",
          r"state[- ]of[- ]the[- ]art", r"real[- ]world", r"proves"]
qc_rows = []
for stem, n_pan in REQUIRED_FIGS.items():
    info = FIGURE_REGISTRY[stem]; p = info["paths"]; img = plt.imread(p["png"])
    text = (info["caption"] + " " + info["texts"]).lower()
    row = {"figure": stem,
           "pdf": p["pdf"].stat().st_size > 5000 and p["pdf"].read_bytes()[:4] == b"%PDF",
           "svg": p["svg"].stat().st_size > 5000,
           "png_dpi_ok": img.shape[1]/info["size_in"][0] >= 290,
           "panels_expected": info["n_panels"] == n_pan,
           # a panel counts as labelled if it has an axis title or (matrix panels) named categorical tick labels
           "axes_labelled": all((a["xlabel"] or a["ylabel"]) or a["named_ticks"] for a in info["axes"]),
           "caption_mentions_temporal_test": "temporal test" in info["caption"].lower(),
           "terminology_ok": not any(re.search(b, text) for b in BANNED),
           "png_px": f"{img.shape[1]}×{img.shape[0]}"}
    row["result"] = "PASS" if all(v for k, v in row.items() if k not in ("figure", "png_px")) else "FAIL"
    qc_rows.append(row)
figure_qc = pd.DataFrame(qc_rows)
figure_qc.to_csv(OUT_DIR/"audit"/"Step7_figure_quality_control.csv", index=False)
FIGURES_OK = bool((figure_qc.result == "PASS").all())
print("FIGURE QUALITY CONTROL\n")
for r in qc_rows:
    print(f"{r['figure'] + ':':48s}{r['result']}  ({r['png_px']})")
print("\n(Aesthetic quality is not machine-verifiable — manual checklist below.)\n")
for item in ["Text readable at final journal width.", "Panel labels consistent.", "Colours consistent across Figures 10–12.",
             "Interpretable in greyscale.", "Legends do not obscure data.", "Axis labels include units.",
             "No misleading axis truncation.", "Counts and CIs match the tables.", "Captions match the analyses.",
             "PDF and SVG open correctly."]:
    print(f"[ ] {item}")
figure_qc


## Figure quality control and manual review

## Secondary IID sensitivity

The temporal protocol remains primary. IID is used only to check whether Step-7 calibration conclusions are highly sensitive to the split design. IID results never alter the temporal policies.

In [ ]:
iid_sensitivity = pd.DataFrame()
if RUN_IID_SENSITIVITY:
    iid_val_all = pd.read_csv(STEP6_DIR/"probabilities"/"iid_validation_probabilities_all9.csv")
    iid_test_all = pd.read_csv(STEP6_DIR/"probabilities"/"iid_test_probabilities_all9.csv")
    iv = model_prob(iid_val_all,REFERENCE_MODEL)
    it = model_prob(iid_test_all,REFERENCE_MODEL)

    m = len(iv)//2
    iv_fit,iv_pol = iv.iloc[:m],iv.iloc[m:]
    cals = fit_calibrators(iv_fit)
    candidates = []
    for name,cal in cals.items():
        pp = apply_calibrator(cal,iv_pol["y_probability_attack"])
        candidates.append({"method":name,**calibration_metrics(iv_pol["y_true"],pp)})
    iid_choice = pd.DataFrame(candidates).sort_values(["brier","ece","nll"]).iloc[0]["method"]

    p_raw = it["y_probability_attack"].to_numpy(float)
    p_cal = apply_calibrator(cals[iid_choice],p_raw)
    yy = it["y_true"].to_numpy(int)
    iid_sensitivity = pd.DataFrame([
        {"protocol":"IID","method":"raw",**calibration_metrics(yy,p_raw)},
        {"protocol":"IID","method":iid_choice,**calibration_metrics(yy,p_cal)}
    ])
    iid_sensitivity.to_csv(OUT_DIR/"tables"/"TableS_Step7_IID_Calibration_Sensitivity.csv",index=False)

iid_sensitivity

In [ ]:
base = classification_metrics(full_test_y,rf_test_raw_p)
cal_raw = calibration_test[calibration_test.method=="raw"].iloc[0]
cal_sel = calibration_test[calibration_test.method==SELECTED_CALIBRATION].iloc[0]

best_net = collaboration_table[
    (collaboration_table["override_policy"]=="confidence-gated profile override") &
    (collaboration_table["target_full_test_budget"]>0)
].sort_values(["net_errors_avoided","macro_f1"],ascending=False).head(1)

evidence = [
    f"The frozen Step-6 Random Forest was retained as the Step-7 reference classifier because it was selected using temporal validation only (validation Macro-F1 {reference['validation_metrics']['Random Forest']['macro_f1']:.4f}).",
    "Calibration models were fitted only on the first chronological half of temporal validation, while calibration-method selection and policy thresholds were determined on the second half. The untouched temporal test set was used only for final evaluation.",
    f"On temporal test, raw RF calibration yielded Brier={cal_raw.brier:.4f} and ECE={cal_raw.ece:.4f}; the validation-selected {SELECTED_CALIBRATION} calibration yielded Brier={cal_sel.brier:.4f} and ECE={cal_sel.ece:.4f}.",
    "Selective-deferral and collaboration experiments use independently learned Step-6 probabilities rather than the generator's `generated_attack_score`, reducing circular dependence on the benchmark-construction score."
]
if len(best_net):
    r = best_net.iloc[0]
    evidence.append(
        f"Under the configured oversight assumptions, the strongest observed confidence-gated routing condition used {r.routing_method} at realized workload {100*r.workload_rate_full_test:.2f}% and produced {int(round(r.net_errors_avoided))} net errors avoided. This is a controlled benchmark result, not evidence of real analyst performance."
    )

text = "# Step 7 manuscript-ready evidence\n\n" + "\n\n".join(evidence)
(OUT_DIR/"manuscript_values"/"step7_revision_evidence.md").write_text(text,encoding="utf-8")
print(text)

In [ ]:
reviewer_lines = [
    (f"Oversight reachability: the review-eligible pool holds {int(reachability_table.iloc[1]['count']):,} of "
     f"{N_TEST:,} temporal-test decisions ({reachability_table.iloc[1]['percent_of_test']:.2f}%), but only "
     f"{int(reachability_table.iloc[4]['count'])} of {int(reachability_table.iloc[2]['count'])} classifier errors fall inside it "
     f"({100*ERROR_COVERAGE:.1f}%). At most {int(reachability_table.iloc[5]['count'])} decisions are correctable by the configured "
     f"profile ({100*CEILING_RATE:.3f}% of all decisions), while {int(reachability_table.iloc[6]['count'])} correct decisions are "
     f"exposed to harm; inside the pool the classifier is correct on {POOL_AI_ACC:.3f} of flows against {POOL_PROFILE_ACC:.3f} for "
     f"the configured profiles. Every override result below is bounded by this ceiling."),
    (f"Generator ambiguity and model uncertainty are largely distinct: model uncertainty identifies pool membership with AUROC "
     f"{auc_pool:.3f} and Spearman correlation {rho:.3f}; only {100*OVERLAP_TOP8:.1f}% of the 8% most model-uncertain flows lie in "
     f"the ambiguity pool. Oversight headroom in HITL-IoT v1.1 is therefore a property of how the review pool was constructed, "
     f"not a limitation of the routing signal."),
    "# Reviewer-facing Step 7 evidence",
    "",
    "## What was added",
    "",
    "1. Independent probability calibration of the frozen validation-selected Random Forest.",
    "2. Selective-prediction/reject-option evaluation at fixed review budgets.",
    "3. Matched-budget random-routing baseline.",
    "4. Human-AI collaboration experiments on the canonical 8% oversight pool.",
    "5. Explicit accounting of AI errors corrected and correct AI decisions harmed.",
    "6. Confidence-gated profile override selected on temporal validation only.",
    "7. Learned deferral utility baseline using out-of-bag training probabilities (when enabled).",
    "8. Secondary IID sensitivity analysis.",
    "9. No use of `generated_attack_score` as the Step-7 AI confidence signal.",
    "",
    "## Interpretation boundary",
    "",
    "The configured oversight profiles are generated benchmark variables. Therefore these results demonstrate that HITL-IoT can support controlled and reproducible comparison of candidate collaboration and oversight mechanisms. They do not establish empirical analyst behaviour, operational SOC effectiveness, professional competence, or legal compliance.",
    "",
    "## Test-set isolation",
    "",
    "No calibration model, confidence gate, review-budget threshold, or learned defer model is fitted or selected using temporal-test labels."
]
reviewer = "\n".join(reviewer_lines)
(OUT_DIR/"manuscript_values"/"reviewer3_step7_evidence.md").write_text(reviewer,encoding="utf-8")
print(reviewer)

In [ ]:
env = {
    "python":sys.version.split()[0],
    "numpy":np.__version__,
    "pandas":pd.__version__,
    "scikit_learn":sklearn.__version__,
    "platform":platform.platform(),
    "seed":SEED,
    "step6_context_fingerprint":context["context_fingerprint"],
    "reference_model":REFERENCE_MODEL,
    "selected_calibration":SELECTED_CALIBRATION,
    "run_all9_calibration":RUN_ALL9_CALIBRATION,
    "run_iid_sensitivity":RUN_IID_SENSITIVITY,
    "run_learned_defer":RUN_LEARNED_DEFER,
}
(OUT_DIR/"logs"/"environment.json").write_text(json.dumps(env,indent=2),encoding="utf-8")

files = [p for p in OUT_DIR.rglob("*") if p.is_file()]
rows = []
for p in sorted(files):
    rows.append({"path":str(p.relative_to(OUT_DIR)),"bytes":p.stat().st_size,"sha256":sha256_file(p)})
pd.DataFrame(rows).to_csv(OUT_DIR/"step7_manifest_sha256.csv",index=False)

summary = {
    "status":"complete",
    "step6_frozen_context":context["context_fingerprint"],
    "reference_model":REFERENCE_MODEL,
    "calibration_method_selected_on_policy_validation":SELECTED_CALIBRATION,
    "temporal_test_n":len(rf_test),
    "oversight_test_n":len(ov_test),
    "oversight_test_rate":len(ov_test)/len(rf_test),
    "profile_confidence_gate_selected_on_validation":PROFILE_GATE,
    "selective_budgets":SELECTIVE_BUDGETS,
    "oversight_budgets_full_test":OVERSIGHT_BUDGETS_FULL_TEST,
}
(OUT_DIR/"step7_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")

print("[PASS] Step-7 manifest written.")

In [ ]:
checks = {
    "Frozen Step-6 context verified": context["context_fingerprint"]==EXPECTED_CONTEXT_FINGERPRINT,
    "Reference model is Random Forest": reference["selected_model"]==REFERENCE_MODEL,
    "Temporal validation contains all 9 models": p_val_all["model"].nunique()==9,
    "Temporal test contains all 9 models": p_test_all["model"].nunique()==9,
    "Validation split into calibration-fit and policy-validation": len(cal_fit)>0 and len(policy_val)>0,
    "Calibration method selected without test labels": SELECTED_CALIBRATION in {"raw","platt","isotonic"},
    "Selective prediction table produced": len(selective_table)>0,
    "Canonical oversight validation/test pools available": len(ov_val)>0 and len(ov_test)>0,
    "Profile confidence gate selected on validation": 0.5<=PROFILE_GATE<=0.95,
    "Oversight mechanism table produced": len(collaboration_table)>0,
    "Generated construction score not used as AI probability": True,
    "Oversight reachability ceiling quantified": len(reachability_table) > 0,
    "Ambiguity vs model-uncertainty overlap quantified": len(overlap_table) > 0,
    "Publication figures 10-12 exported and QC-checked": FIGURES_OK,
    "Manifest produced": (OUT_DIR/"step7_manifest_sha256.csv").exists(),
}

print("="*78)
print("HITL-IoT v1.1 — STEP 7 SELECTIVE DEFERRAL / CALIBRATION / OVERSIGHT")
print("="*78)
for k,v in checks.items():
    print(f"{k+':':58s} {'PASS' if v else 'FAIL'}")
print("\nPrimary protocol: temporal")
print("Reference model:",REFERENCE_MODEL)
print("Selected calibration:",SELECTED_CALIBRATION)
print("Oversight test pool:",len(ov_test),f"({len(ov_test)/len(rf_test):.2%})")
print(f"Oversight reachability ceiling: {int(reachability_table.iloc[5]['count'])} correctable decisions "
      f"({100*CEILING_RATE:.3f}% of the temporal test), harm exposure {int(reachability_table.iloc[6]['count'])}")
print("\nNEXT: freeze Step 7, then proceed to Step 8 external validation with CICIoT2023 + ToN-IoT Network.")
assert all(checks.values()), [k for k,v in checks.items() if not v]

## Interpretation for the manuscript

The strongest defensible claim from Step 7 is:

> HITL-IoT supports controlled and reproducible comparison of calibration, selective-deferral, review-allocation, override, and learned-routing mechanisms using independently learned IDS probabilities and explicitly configured oversight assumptions.

Do **not** convert this into claims that the configured profiles reproduce real security analysts.

After Step 7 is frozen, proceed to:

**`05_HITL-IoT_v1.1_External_Validation.ipynb`**

using **CICIoT2023** as the primary external dataset and **ToN-IoT Network** as the second external dataset.